## House Prices (Data Preparation — Structure)

In [ ]:
from IPython.display import display
from house_prices_mlops.pipeline import load_stage
from house_prices_mlops.data_tracker import DataTracker
import pandas as pd

df_before = load_stage("train")
tracker = DataTracker.load(df_before)
display(tracker.profile)

2026-10-11 20:42:37.321 | INFO     | house_prices_mlops.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops


### Structure

- Adjust physical data types.
- Review conceptual classifications when needed.
- Review temporal representations.
- Keep identifiers for traceability and exclude them from model predictors.

The original DP_Structure implements only the conversion of MSSubClass codes to text. Other items remain topics for review. No intermediate dataset is saved.

### Findings

In [3]:
structure_findings = tracker.findings.loc[
    tracker.findings["key"].eq("ms_subclass_representation")
].copy()
display(structure_findings)


,key,var,stage,finding,status,resolution
15,ms_subclass_representation,MSSubClass,DU_Structure,Variable nominal representada mediante códigos...,pending,None


### MSSubClass Representation

In [4]:
from house_prices_mlops.preparation.structure import normalize_ms_subclass

before_dtype = str(df_before["MSSubClass"].dtype)
df_individual = normalize_ms_subclass(df_before)
df_hp = load_stage("structure")
pd.testing.assert_frame_equal(df_individual, df_hp)
after_dtype = str(df_hp["MSSubClass"].dtype)

print("Before:", before_dtype)
print("After:", after_dtype)
display(df_hp[["Id", "MSSubClass"]].head())


Before: int64
After: string


,Id,MSSubClass
0,1,60
1,2,20
2,3,60
3,4,70
4,5,60


### Validation

In [5]:


assert len(df_hp) == len(df_before)
pd.testing.assert_frame_equal(
    df_hp.drop(columns="MSSubClass"),
    df_before.drop(columns="MSSubClass"),
)
pd.testing.assert_series_equal(
    df_before["MSSubClass"], load_stage("train")["MSSubClass"]
)
tracker.update_profile(df_hp)
display(tracker.profile.loc[tracker.profile["var"].eq("MSSubClass")])


,var,data_type,variable_type,unique_count,missing_count,missing_pct,zero_count,zero_pct,min,max,mean,median,std,skewness,kurtosis
1,MSSubClass,string,nominal,15,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Record Decision

In [6]:
tracker.log(
    var="MSSubClass",
    stage="DP - Structure",
    action="change_dtype",
    method='astype("string")',
    before=before_dtype,
    after=after_dtype,
)

tracker.resolve_finding(
    key="ms_subclass_representation",
    resolution="MSSubClass se convirtió de códigos numéricos a texto para preservar su naturaleza nominal.",
)


### Save Findings

In [7]:
tracker.save()
